# Snowflake Serving Layer

This notebook publishes the processed datasets generated from the Spark pipeline to Snowflake.

The data supports a location scoring model designed to identify promising census tracts for potential Whole Foods store expansion.

Two warehouse tables are created:
- TRACT_MASTER
- TOP_CANDIDATES



In [1]:
# Install Snowflake connector
!pip install "snowflake-connector-python[pandas]"

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Step 1: Import packages and define paths

import os
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

BASE_PATH = "/content/drive/MyDrive/msba405-group-project/"
PROCESSED_PATH = BASE_PATH + "processed_data/"

TRACT_MASTER_PATH = PROCESSED_PATH + "tract_master.csv"
TOP_CANDIDATES_PATH = PROCESSED_PATH + "top_candidates.csv"

print("TRACT_MASTER_PATH:", TRACT_MASTER_PATH)
print("TOP_CANDIDATES_PATH:", TOP_CANDIDATES_PATH)

print("tract_master exists:", os.path.exists(TRACT_MASTER_PATH))
print("top_candidates exists:", os.path.exists(TOP_CANDIDATES_PATH))

TRACT_MASTER_PATH: /content/drive/MyDrive/msba405-group-project/processed_data/tract_master.csv
TOP_CANDIDATES_PATH: /content/drive/MyDrive/msba405-group-project/processed_data/top_candidates.csv
tract_master exists: True
top_candidates exists: True


In [4]:
# Step 2: Load processed CSV files

tract_master = pd.read_csv(TRACT_MASTER_PATH)
top_candidates = pd.read_csv(TOP_CANDIDATES_PATH)

print("tract_master shape:", tract_master.shape)
print("top_candidates shape:", top_candidates.shape)

display(tract_master.head())
display(top_candidates.head())

tract_master shape: (2498, 9)
top_candidates shape: (10, 7)


,GEOID,income,ba_rate,pop_density,retail_density,crime_low_60,has_wf,distance,score
0,6037101110,80500.333333,0.283800,3635.617733,0.0,False,0,2.459528,0.289057
1,6037101122,107695.000000,0.367207,1574.103315,0.0,True,0,2.053820,0.327459
2,6037101220,75710.666667,0.323667,4951.689784,0.0,False,0,2.409450,0.293302
3,6037101221,53116.666667,0.262423,10616.130872,0.0,False,0,3.260561,0.234711
4,6037101222,40188.666667,0.159255,8622.472840,0.0,False,0,3.574630,0.218597


,GEOID,income,ba_rate,pop_density,retail_density,distance,score
0,6037620601,123234.0,0.572,4365.8,20.50,0.263,0.792
1,6037480704,114216.0,0.652,3824.9,16.83,0.364,0.733
2,6037702201,110573.0,0.669,3329.4,22.42,0.458,0.686
3,6037700802,116588.0,0.599,5924.1,17.51,0.473,0.679
4,6037310300,136227.0,0.597,2860.6,21.67,0.525,0.656


In [5]:
# Step 3: Standardize column names before upload

tract_master.columns = tract_master.columns.str.upper()
top_candidates.columns = top_candidates.columns.str.upper()

print("TRACT_MASTER columns:")
print(tract_master.columns.tolist())

print("\nTOP_CANDIDATES columns:")
print(top_candidates.columns.tolist())

TRACT_MASTER columns:
['GEOID', 'INCOME', 'BA_RATE', 'POP_DENSITY', 'RETAIL_DENSITY', 'CRIME_LOW_60', 'HAS_WF', 'DISTANCE', 'SCORE']

TOP_CANDIDATES columns:
['GEOID', 'INCOME', 'BA_RATE', 'POP_DENSITY', 'RETAIL_DENSITY', 'DISTANCE', 'SCORE']


In [6]:
# Step 4: Load Snowflake password from .env

!pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/msba405-group-project/.env")

password = os.getenv("SNOWFLAKE_PASSWORD")

print("Password loaded:", password is not None)

Password loaded: True


In [7]:
# Step 5: Connect to Snowflake

conn = snowflake.connector.connect(
    user="NUOCID",
    password=password,
    account="PWAUQJP-VRB70333",
    warehouse="COMPUTE_WH",
    database="MSBA405",
    schema="PROJECT"
)

cursor = conn.cursor()

print("Snowflake connected successfully")

Snowflake connected successfully


In [8]:
# Step 6: Upload data to Snowflake

success1, nchunks1, nrows1, _ = write_pandas(
    conn,
    tract_master,
    "TRACT_MASTER",
    auto_create_table=True,
    overwrite=True
)

success2, nchunks2, nrows2, _ = write_pandas(
    conn,
    top_candidates,
    "TOP_CANDIDATES",
    auto_create_table=True,
    overwrite=True
)

print("TRACT_MASTER uploaded:", success1, "rows:", nrows1)
print("TOP_CANDIDATES uploaded:", success2, "rows:", nrows2)

TRACT_MASTER uploaded: True rows: 2498
TOP_CANDIDATES uploaded: True rows: 10


In [9]:
# Step 7: Verify uploaded tables

cursor.execute("SELECT COUNT(*) FROM TRACT_MASTER")
print("TRACT_MASTER rows:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM TOP_CANDIDATES")
print("TOP_CANDIDATES rows:", cursor.fetchone()[0])

TRACT_MASTER rows: 2498
TOP_CANDIDATES rows: 10


In [10]:
# Step 8: Check Snowflake schemas

cursor.execute("DESC TABLE TRACT_MASTER")
print("TRACT_MASTER schema:")
for row in cursor.fetchall():
    print(row)

print("\n" + "-" * 60 + "\n")

cursor.execute("DESC TABLE TOP_CANDIDATES")
print("TOP_CANDIDATES schema:")
for row in cursor.fetchall():
    print(row)

TRACT_MASTER schema:
('GEOID', 'NUMBER(38,0)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('INCOME', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('BA_RATE', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('POP_DENSITY', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('RETAIL_DENSITY', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('CRIME_LOW_60', 'BOOLEAN', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('HAS_WF', 'NUMBER(38,0)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('DISTANCE', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('SCORE', 'FLOAT', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)

------------------------------------------------------------

TOP_CANDIDATES schema:
('GEOID', 'NUMBER(38,0)', 'COLUMN', 'Y', None, 'N', 'N', None, None, None, None, None)
('INCOME', 'FLOAT', 'COLUMN', 'Y', None, 

In [11]:
# Step 9: Preview tables

cursor.execute("""
SELECT *
FROM TRACT_MASTER
LIMIT 5
""")

print("TRACT_MASTER preview:")
for row in cursor.fetchall():
    print(row)

print("\n" + "-" * 60 + "\n")

cursor.execute("""
SELECT *
FROM TOP_CANDIDATES
ORDER BY SCORE DESC
LIMIT 5
""")

print("TOP_CANDIDATES preview:")
for row in cursor.fetchall():
    print(row)

TRACT_MASTER preview:
(6037101110, 80500.33333333333, 0.2838001954867394, 3635.617732594188, 0.0, False, 0, 2.4595282636626097, 0.2890567510326676)
(6037101122, 107695.0, 0.3672074949333901, 1574.103315185634, 0.0, True, 0, 2.0538200002874545, 0.327458723797038)
(6037101220, 75710.66666666667, 0.3236673625193479, 4951.689784409284, 0.0, False, 0, 2.4094497622109894, 0.2933024592658938)
(6037101221, 53116.66666666666, 0.262423083395207, 10616.130872047388, 0.0, False, 0, 3.2605609278449115, 0.2347108789043471)
(6037101222, 40188.66666666666, 0.1592546043761676, 8622.472839772645, 0.0, False, 0, 3.574629755835798, 0.2185969255160622)

------------------------------------------------------------

TOP_CANDIDATES preview:
(6037620601, 123234.0, 0.572, 4365.8, 20.5, 0.263, 0.792)
(6037480704, 114216.0, 0.652, 3824.9, 16.83, 0.364, 0.733)
(6037702201, 110573.0, 0.669, 3329.4, 22.42, 0.458, 0.686)
(6037700802, 116588.0, 0.599, 5924.1, 17.51, 0.473, 0.679)
(6037310300, 136227.0, 0.597, 2860.6, 

## Final Recommendation Query

The `TOP_CANDIDATES` table contains the final ranking results produced by the Whole Foods location scoring model.

Each census tract is scored based on demographic, retail, and spatial features generated in the Spark pipeline.  

The query below retrieves the highest scoring census tracts, representing the most promising candidate locations for potential Whole Foods expansion.

In [12]:
# Step 10: Show final ranked recommendation

cursor.execute("""
SELECT
    GEOID,
    INCOME,
    BA_RATE,
    POP_DENSITY,
    RETAIL_DENSITY,
    DISTANCE,
    SCORE
FROM TOP_CANDIDATES
ORDER BY SCORE DESC
""")

print("Top recommended census tracts:\n")
for row in cursor.fetchall():
    print(row)

Top recommended census tracts:

(6037620601, 123234.0, 0.572, 4365.8, 20.5, 0.263, 0.792)
(6037480704, 114216.0, 0.652, 3824.9, 16.83, 0.364, 0.733)
(6037702201, 110573.0, 0.669, 3329.4, 22.42, 0.458, 0.686)
(6037700802, 116588.0, 0.599, 5924.1, 17.51, 0.473, 0.679)
(6037310300, 136227.0, 0.597, 2860.6, 21.67, 0.525, 0.656)
(6037621301, 117529.0, 0.64, 4575.9, 12.8, 0.569, 0.637)
(6037700901, 105686.0, 0.696, 3960.8, 18.03, 0.573, 0.636)
(6037621324, 114294.0, 0.719, 5615.9, 17.2, 0.602, 0.624)
(6037577200, 108071.0, 0.62, 5794.2, 14.24, 0.681, 0.595)
(6037554519, 121399.0, 0.562, 1995.2, 15.82, 0.686, 0.593)


In [13]:
# Step 11: Close Snowflake connection

cursor.close()
conn.close()

print("Snowflake connection closed.")

Snowflake connection closed.
